# Ålands bussnät – GTFS-utforskning och (pausad) optimering

Notebooken har två delar:

- **Del 1 – GTFS-utforskning** (aktiv): ladda upp mappen med dina **GTFS-filer** via en knapp, se hela nätet på en
  interaktiv karta, klicka på en linje för **avgångar/dag**, **avgångar/timme i förmiddags- (06–09) och
  eftermiddagsrusning (15–18)** samt **riktning**, och välj ut de linjer du vill arbeta vidare med.
- **Del 2 – Optimering med genetisk algoritm** (pausad): den tidigare optimeringen. Den körs bara om du sätter
  `KOR_OPTIMERING = True`.

Börja med Del 1 nedan.

## (Valfritt) Google Drive i Colab

**Del 1 (GTFS) behöver INTE detta** – du laddar upp din GTFS-mapp med knappen längre ner. Kör bara den här cellen
om du vill spara filer i din Google Drive eller köra **Del 2** (optimeringen) därifrån. Sätt då
`MONTERA_DRIVE = True`. Misslyckas monteringen stoppas inte notebooken – Del 1 fungerar ändå.

In [ ]:
import sys, os

MONTERA_DRIVE = False   # sätt True endast om du vill använda Google Drive / köra Del 2 därifrån

if 'google.colab' in sys.modules and MONTERA_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        REPO_URL = 'https://github.com/JesperHelen/-land.git'   # publikt repo – ingen nyckel behövs
        REPO_DIR = '/content/drive/MyDrive/aland_ga'
        if os.path.isdir(os.path.join(REPO_DIR, '.git')):
            os.system(f'git -C "{REPO_DIR}" pull --ff-only')
        else:
            os.system(f'git clone "{REPO_URL}" "{REPO_DIR}"')
        os.chdir(REPO_DIR)
        print('Drive monterad. Arbetskatalog:', os.getcwd())
    except Exception as e:
        print('Kunde inte montera Drive/klona (', e, ') – hoppar över. Del 1 fungerar ändå via mappväljaren nedan.')
elif 'google.colab' in sys.modules:
    print('Colab: hoppar över Drive-montering (behövs inte för Del 1). Ladda upp din GTFS-mapp med knappen nedan.')
else:
    print('Inte i Google Colab – kör lokalt som vanligt.')

## Beroenden
Installera vid behov med `pip install -r requirements.txt`.

In [ ]:
import os, sys, json, math, time, random, datetime, zipfile, tempfile, base64
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra

try:
    import folium
    from folium.plugins import PolyLineTextPath
    HAR_FOLIUM = True
except Exception:
    HAR_FOLIUM = False
    print('Folium saknas – kartorna ritas statiskt. Installera med: pip install folium')

try:
    from tqdm.auto import tqdm
except Exception:               # tqdm är valfritt
    def tqdm(x, **k):
        return x

# Del 1 – GTFS-utforskning

Läs in GTFS, se hela nätet och välj vilka linjer du vill arbeta vidare med.

### GTFS-funktioner
Inläsning, linjestatistik och kartor. (Behöver normalt inte ändras.)

In [ ]:
GTFS_FILER = ['agency', 'stops', 'routes', 'trips', 'stop_times',
              'calendar', 'calendar_dates', 'shapes', 'frequencies']

def las_gtfs(path):
    """Läser en GTFS-mapp ELLER .zip till en dict av DataFrames (allt som text)."""
    tab = {}
    if str(path).lower().endswith('.zip'):
        with zipfile.ZipFile(path) as zf:
            namn = {os.path.basename(n).replace('.txt', ''): n
                    for n in zf.namelist() if n.endswith('.txt')}
            for t in GTFS_FILER:
                if t in namn:
                    with zf.open(namn[t]) as fh:
                        tab[t] = pd.read_csv(fh, dtype=str, keep_default_na=False)
    else:
        for t in GTFS_FILER:
            fp = os.path.join(path, t + '.txt')
            if os.path.exists(fp):
                tab[t] = pd.read_csv(fp, dtype=str, keep_default_na=False)
    if 'stops' not in tab or 'routes' not in tab or 'trips' not in tab or 'stop_times' not in tab:
        raise FileNotFoundError('GTFS ofullständig: stops/routes/trips/stop_times krävs. Hittade: '
                                + ', '.join(sorted(tab)))
    # numeriska kolumner
    for c in ['stop_lat', 'stop_lon']:
        tab['stops'][c] = pd.to_numeric(tab['stops'][c], errors='coerce')
    tab['stop_times']['stop_sequence'] = pd.to_numeric(tab['stop_times']['stop_sequence'], errors='coerce')
    if 'direction_id' not in tab['trips'].columns:
        tab['trips']['direction_id'] = '0'
    tab['trips']['direction_id'] = tab['trips']['direction_id'].replace('', '0')
    return tab

def gtfs_tid_min(s):
    """GTFS-tid 'HH:MM:SS' (kan vara >24h) -> minuter efter midnatt (float)."""
    if not s or ':' not in str(s):
        return np.nan
    d = str(s).split(':')
    return int(d[0])*60 + int(d[1]) + (int(d[2])/60 if len(d) > 2 else 0)

# ------------------------------------------------------------------ aktiva turer en viss dag
def aktiva_services(gtfs, datum):
    """service_id:n som trafikeras ett visst datum (datetime.date). Hanterar calendar + calendar_dates."""
    aktiva = set()
    vd = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday'][datum.weekday()]
    dstr = datum.strftime('%Y%m%d')
    if 'calendar' in gtfs:
        cal = gtfs['calendar']
        for _, r in cal.iterrows():
            if r.get(vd, '0') == '1' and r.get('start_date', '00000000') <= dstr <= r.get('end_date', '99999999'):
                aktiva.add(r['service_id'])
    if 'calendar_dates' in gtfs:
        for _, r in gtfs['calendar_dates'].iterrows():
            if r['date'] == dstr:
                if r['exception_type'] == '1':
                    aktiva.add(r['service_id'])
                elif r['exception_type'] == '2':
                    aktiva.discard(r['service_id'])
    return aktiva

def valj_standarddatum(gtfs):
    """Föreslår ett representativt vardagsdatum som faktiskt har trafik."""
    idag = datetime.date.today()
    for off in range(0, 21):
        d = idag + datetime.timedelta(days=off)
        if d.weekday() < 5 and aktiva_services(gtfs, d):
            return d
    # annars: ta första giltiga datum i calendar
    if 'calendar' in gtfs and len(gtfs['calendar']):
        s = gtfs['calendar']['start_date'].min()
        return datetime.datetime.strptime(s, '%Y%m%d').date()
    return idag

# ------------------------------------------------------------------ turernas första avgång
def trip_forsta_avgang(gtfs):
    """DataFrame: en rad per trip med route_id, direction_id, service_id och avgångstid (min) vid första hållplats."""
    st = gtfs['stop_times'].copy()
    st['dep_min'] = st['departure_time'].map(gtfs_tid_min)
    forsta = st.sort_values('stop_sequence').groupby('trip_id', as_index=False).first()[['trip_id', 'dep_min']]
    tr = gtfs['trips'][['trip_id', 'route_id', 'direction_id', 'service_id']]
    return tr.merge(forsta, on='trip_id', how='left')

# ------------------------------------------------------------------ linjestatistik
def linje_statistik(gtfs, datum, fm=(6, 9), em=(15, 18)):
    """Per linje: avgångar/dag, avgångar/timme i FM- resp EM-rusning, per riktning + riktningsstatus."""
    aktiva = aktiva_services(gtfs, datum)
    tf = trip_forsta_avgang(gtfs)
    tf = tf[tf['service_id'].isin(aktiva)]
    routes = gtfs['routes']
    namn = {r['route_id']: (r.get('route_short_name') or r.get('route_id'),
                            r.get('route_long_name', '')) for _, r in routes.iterrows()}
    fm_lo, fm_hi = fm[0]*60, fm[1]*60
    em_lo, em_hi = em[0]*60, em[1]*60
    fm_h = max(1, fm[1]-fm[0]); em_h = max(1, em[1]-em[0])
    stat = {}
    for rid in routes['route_id']:
        sub = tf[tf['route_id'] == rid]
        per_dir = {}
        for d in sorted(sub['direction_id'].unique()):
            s = sub[sub['direction_id'] == d]
            dep = s['dep_min'].dropna()
            per_dir[d] = {
                'avg_dag': int(len(s)),
                'fm_antal': int(((dep >= fm_lo) & (dep < fm_hi)).sum()),
                'em_antal': int(((dep >= em_lo) & (dep < em_hi)).sum()),
                'forsta': _mmhh(dep.min()) if len(dep) else '-',
                'sista': _mmhh(dep.max()) if len(dep) else '-',
            }
            per_dir[d]['fm_per_h'] = round(per_dir[d]['fm_antal']/fm_h, 1)
            per_dir[d]['em_per_h'] = round(per_dir[d]['em_antal']/em_h, 1)
        stat[rid] = {
            'kort': namn[rid][0], 'lang': namn[rid][1],
            'avg_dag': int(len(sub)),
            'riktningar': sorted(sub['direction_id'].unique()),
            'bada_riktningar': len(sub['direction_id'].unique()) >= 2,
            'per_dir': per_dir,
        }
    return stat

def _mmhh(m):
    if m is None or (isinstance(m, float) and math.isnan(m)):
        return '-'
    m = int(round(m)); return f'{(m//60)%24:02d}:{m%60:02d}'

In [ ]:
def linje_geometri(gtfs, route_id, direction):
    """Koordinatsekvens (lat,lon) för en linje+riktning – från shapes om möjligt, annars hållplatsföljden."""
    trips = gtfs['trips']
    tsub = trips[(trips['route_id'] == route_id) & (trips['direction_id'] == str(direction))]
    if not len(tsub):
        return []
    if 'shapes' in gtfs and 'shape_id' in tsub.columns and tsub.iloc[0].get('shape_id', ''):
        sid = tsub.iloc[0]['shape_id']
        s = gtfs['shapes']
        s = s[s['shape_id'] == sid].copy()
        if len(s):
            s['seq'] = pd.to_numeric(s['shape_pt_sequence'], errors='coerce')
            s = s.sort_values('seq')
            return list(zip(s['shape_pt_lat'].astype(float), s['shape_pt_lon'].astype(float)))
    # fallback: hållplatsföljd från en representativ trip
    tid = tsub.iloc[0]['trip_id']
    st = gtfs['stop_times']
    st = st[st['trip_id'] == tid].sort_values('stop_sequence')
    stops = gtfs['stops'].set_index('stop_id')
    coords = []
    for sid_ in st['stop_id']:
        if sid_ in stops.index:
            coords.append((float(stops.loc[sid_, 'stop_lat']), float(stops.loc[sid_, 'stop_lon'])))
    return coords

def _offset_linje(coords, meter):
    """Förskjuter en linje vinkelrätt `meter` meter (så båda riktningar syns bredvid varandra)."""
    if len(coords) < 2 or not meter:
        return [list(c) for c in coords]
    out = []
    for i in range(len(coords)):
        la, lo = coords[i]
        a = coords[max(0, i-1)]; b = coords[min(len(coords)-1, i+1)]
        mlat = 111320.0; mlon = 111320.0*math.cos(math.radians(la))
        vx = (b[1]-a[1])*mlon; vy = (b[0]-a[0])*mlat
        n = math.hypot(vx, vy) or 1.0
        pvx, pvy = -vy/n, vx/n                      # perpendikel (meter)
        out.append([la + (meter*pvy)/mlat, lo + (meter*pvx)/mlon])
    return out

# ------------------------------------------------------------------ interaktiv karta
FARGER = ['#e6194B', '#3cb44b', '#4363d8', '#911eb4', '#f58231', '#469990',
          '#800000', '#000075', '#9A6324', '#808000', '#f032e6', '#42d4f4']

def _popup_html(s):
    rows = ''
    for d in s.get('riktningar', []):
        pd_ = s['per_dir'][d]
        rikt = 'utåt (0)' if d == '0' else ('retur (1)' if d == '1' else f'riktn {d}')
        rows += (f"<tr><td>{rikt}</td><td style='text-align:center'>{pd_['avg_dag']}</td>"
                 f"<td style='text-align:center'>{pd_['fm_per_h']}/h</td>"
                 f"<td style='text-align:center'>{pd_['em_per_h']}/h</td>"
                 f"<td>{pd_['forsta']}–{pd_['sista']}</td></tr>")
    rikt_txt = 'Båda riktningar' if s.get('bada_riktningar') else 'En riktning'
    return (f"<div style='font-family:sans-serif'><b>Linje {s.get('kort','')}</b> – {s.get('lang','')}<br>"
            f"<b>{rikt_txt}</b> · {s.get('avg_dag',0)} avgångar/dag"
            f"<table border='1' cellpadding='3' style='border-collapse:collapse;font-size:12px;margin-top:4px'>"
            f"<tr><th>Riktning</th><th>Avg/dag</th><th>FM 6–9</th><th>EM 15–18</th><th>Första–sista</th></tr>"
            f"{rows}</table></div>")

def rita_natverk(gtfs, stat, valda=None, filnamn=None, visa_pilar=True, titel='Ålands bussnät (GTFS)'):
    """Interaktiv Folium-karta över hela nätet. Valda linjer framhävs; klick på linje visar statistik + riktning."""
    import folium
    from folium.plugins import PolyLineTextPath
    valda = set(valda or [])
    stops = gtfs['stops']
    m = folium.Map(location=[float(stops['stop_lat'].mean()), float(stops['stop_lon'].mean())],
                   zoom_start=10, tiles='OpenStreetMap', control_scale=True)
    routes = list(gtfs['routes']['route_id'])
    for idx, rid in enumerate(routes):
        if valda and rid not in valda:
            continue                     # visa ENDAST valda linjer (tomt urval = hela nätet)
        s = stat.get(rid, {})
        color = FARGER[idx % len(FARGER)]
        opacity = 0.9
        weight = 5
        namn = f"Linje {s.get('kort', rid)}"
        fg = folium.FeatureGroup(name=namn, show=True)
        for d in s.get('riktningar', ['0']):
            coords = linje_geometri(gtfs, rid, d)
            if len(coords) < 2:
                continue
            off = 14 if d == '1' else (-14 if s.get('bada_riktningar') else 0)
            coords2 = _offset_linje(coords, off)
            pl = folium.PolyLine(coords2, color=color, weight=weight, opacity=opacity,
                                 popup=folium.Popup(_popup_html(s), max_width=340), tooltip=namn)
            pl.add_to(fg)
            if visa_pilar:
                PolyLineTextPath(pl, '  ▶  ', repeat=True, offset=7,
                                 attributes={'fill': color, 'font-weight': 'bold', 'font-size': '15'}).add_to(fg)
        fg.add_to(m)
    folium.LayerControl(collapsed=True).add_to(m)
    titel_html = (f"<div style='position:fixed;top:10px;left:60px;z-index:9999;background:white;"
                  f"padding:6px 10px;border-radius:6px;font-family:sans-serif;font-size:14px;"
                  f"box-shadow:0 1px 4px rgba(0,0,0,.3)'><b>{titel}</b><br>"
                  f"<span style='font-size:11px'>▶ = färdriktning · klicka på en linje för statistik</span></div>")
    m.get_root().html.add_child(folium.Element(titel_html))
    if filnamn:
        m.save(filnamn)
    return m

def rita_natverk_statisk(gtfs, stat, valda=None, filnamn=None, titel='Ålands bussnät (GTFS)'):
    """Statisk matplotlib-karta med riktningspilar (reserv om Folium ej kan visas)."""
    import matplotlib.pyplot as plt
    valda = set(valda or [])
    fig, ax = plt.subplots(figsize=(12, 10))
    routes = list(gtfs['routes']['route_id'])
    for idx, rid in enumerate(routes):
        if valda and rid not in valda:
            continue                     # visa ENDAST valda linjer (tomt urval = hela nätet)
        s = stat.get(rid, {}); color = FARGER[idx % len(FARGER)]
        lw = 3.0
        alpha = 0.95
        for k, d in enumerate(s.get('riktningar', ['0'])):
            coords = linje_geometri(gtfs, rid, d)
            if len(coords) < 2:
                continue
            off = 120 if d == '1' else (-120 if s.get('bada_riktningar') else 0)
            c = _offset_linje(coords, off)
            lat = [p[0] for p in c]; lon = [p[1] for p in c]
            lbl = f"Linje {s.get('kort', rid)}" if k == 0 else None
            ax.plot(lon, lat, '-', color=color, lw=lw, alpha=alpha, label=lbl, zorder=3)
            for frac in (0.3, 0.62):                       # riktningspilar
                i = int(frac*(len(c)-1)); j = min(i+1, len(c)-1)
                if j > i:
                    ax.annotate('', xy=(lon[j], lat[j]), xytext=(lon[i], lat[i]),
                                arrowprops=dict(arrowstyle='-|>', color=color, lw=lw, alpha=alpha), zorder=4)
    ax.scatter(gtfs['stops']['stop_lon'], gtfs['stops']['stop_lat'], s=6, c='#999', zorder=1)
    ax.set_aspect(1/math.cos(math.radians(float(gtfs['stops']['stop_lat'].mean()))))
    ax.set_title(titel); ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud')
    ax.legend(loc='upper right', fontsize=8, framealpha=0.9)
    plt.tight_layout()
    if filnamn:
        plt.savefig(filnamn, dpi=130)
    plt.show()

## Steg 1 – Välj mappen med GTFS-filerna
Klicka på **Välj GTFS-mapp** och peka ut mappen som innehåller dina GTFS-filer (`stops.txt`, `routes.txt`,
`trips.txt`, `stop_times.txt`, …). I Google Colab läses hela mappen in direkt i webbläsaren. Utanför Colab kan du
i stället använda uppladdningsknappen (markera filerna) eller sätta `GTFS_PATH` manuellt.

> Vill du bara testa? Kör cellen och tryck på **Använd exempeldata (Åland)** så laddas det medföljande exemplet.

In [ ]:
gtfs = None
GTFS_PATH = ''   # kan sättas manuellt till en mapp eller .zip

def ladda_gtfs(path):
    # Läser in GTFS från en mapp/zip och sparar i globala `gtfs`.
    global gtfs, GTFS_PATH
    GTFS_PATH = path
    gtfs = las_gtfs(path)
    print('✔ GTFS inläst från', path, '→', {k: len(v) for k, v in gtfs.items()})

def _skriv_temp(filer):
    # filer: lista av (namn, bytes) -> skriver till temp-mapp och läser in.
    d = tempfile.mkdtemp(prefix='gtfs_')
    for namn, data in filer:
        if namn.lower().endswith('.txt'):
            with open(os.path.join(d, os.path.basename(namn)), 'wb') as fh:
                fh.write(data)
    ladda_gtfs(d)

def anvand_exempeldata():
    for kand in ['data/gtfs_sample', '../data/gtfs_sample', 'gtfs_sample']:
        if os.path.exists(kand):
            ladda_gtfs(kand); return
    print('Hittade inget exempel (data/gtfs_sample).')

if 'google.colab' in sys.modules:
    # Riktig mappväljare i webbläsaren (webkitdirectory) -> skickar filerna till kärnan
    from google.colab import output as _cout
    from IPython.display import display, HTML
    def _ta_emot_gtfs(filer):
        _skriv_temp([(f['name'], base64.b64decode(f['content'])) for f in filer])
        return {}
    _cout.register_callback('notebook.ta_emot_gtfs', _ta_emot_gtfs)
    display(HTML('''
      <button onclick="document.getElementById('gtfsdir').click()"
              style="padding:8px 14px;font-size:14px;border:0;border-radius:6px;background:#1a73e8;color:#fff;cursor:pointer">
        📁 Välj GTFS-mapp</button>
      <input type="file" webkitdirectory directory multiple id="gtfsdir" style="display:none">
      <span id="gtfsmsg" style="margin-left:10px;font-family:sans-serif"></span>
      <script>
        const inp = document.getElementById('gtfsdir');
        inp.addEventListener('change', async (e) => {
          const msg = document.getElementById('gtfsmsg'); msg.textContent = 'Läser filer...';
          const ut = [];
          for (const f of e.target.files) {
            if (!f.name.toLowerCase().endsWith('.txt')) continue;
            const buf = await f.arrayBuffer();
            let bin = ''; const bytes = new Uint8Array(buf);
            for (let i = 0; i < bytes.length; i++) bin += String.fromCharCode(bytes[i]);
            ut.push({name: f.name, content: btoa(bin)});
          }
          msg.textContent = 'Skickar ' + ut.length + ' filer till Python...';
          await google.colab.kernel.invokeFunction('notebook.ta_emot_gtfs', [ut], {});
          msg.textContent = '✔ Inläst (' + ut.length + ' filer). Kör nästa cell.';
        });
      </script>
    '''))
    print('Klicka på "Välj GTFS-mapp" och välj din GTFS-mapp. Vänta på "✔ Inläst" och kör sedan nästa cell.')
    print('(För att testa utan egen data: kör  anvand_exempeldata()  i en cell.)')
else:
    # Lokalt/Jupyter: uppladdningsknapp (markera GTFS-filerna eller en .zip), annars manuell GTFS_PATH
    try:
        import ipywidgets as widgets
        from IPython.display import display
        _upp = widgets.FileUpload(accept='.txt,.zip', multiple=True, description='📁 Välj GTFS-filer')
        def _on_upp(change):
            val = _upp.value
            poster = (val.values() if isinstance(val, dict) else val)   # v7 dict / v8 tuple
            filer = []
            for meta in poster:
                namn = meta['metadata']['name'] if 'metadata' in meta else meta['name']
                filer.append((namn, bytes(meta['content'])))
            if filer and filer[0][0].lower().endswith('.zip'):
                d = tempfile.mkdtemp(prefix='gtfs_'); zp = os.path.join(d, filer[0][0])
                open(zp, 'wb').write(filer[0][1]); ladda_gtfs(zp)
            else:
                _skriv_temp(filer)
        _upp.observe(_on_upp, names='value')
        display(_upp)
        print('Markera dina GTFS-filer (eller en .zip). Eller kör  anvand_exempeldata()  för exempel.')
    except Exception:
        print('Sätt GTFS_PATH till din GTFS-mapp/.zip och kör  ladda_gtfs(GTFS_PATH)  – eller  anvand_exempeldata().')

# Om inget laddats interaktivt (t.ex. vid \"Kör alla\" lokalt) – ta exempeldata som reserv
if gtfs is None and 'google.colab' not in sys.modules and os.path.exists('data/gtfs_sample'):
    anvand_exempeldata()

### Utdatamapp

In [ ]:
UT = next((p for p in ['output', '../output'] if os.path.isdir(p)), '.')
os.makedirs(UT, exist_ok=True)
assert gtfs is not None, 'Ingen GTFS inläst ännu – välj mapp via knappen ovan (vänta på \"✔ Inläst\") eller kör anvand_exempeldata().'
print('GTFS:', {k: len(v) for k, v in gtfs.items()}, '| utdatamapp:', UT)

## Steg 2 – Trafikdygn och statistik
Statistiken beräknas för ett **vardagsdygn** (väljs automatiskt). Rusningsfönstren är förmiddag **06–09** och
eftermiddag **15–18** (ändra vid behov).

In [ ]:
DATUM      = ''          # '' => välj automatiskt ett vardagsdatum med trafik; annars 'ÅÅÅÅ-MM-DD'
FM_RUSNING = (6, 9)      # förmiddagsrusning
EM_RUSNING = (15, 18)    # eftermiddagsrusning

datum = datetime.date.fromisoformat(DATUM) if DATUM else valj_standarddatum(gtfs)
veckodag = ['måndag', 'tisdag', 'onsdag', 'torsdag', 'fredag', 'lördag', 'söndag'][datum.weekday()]
stat = linje_statistik(gtfs, datum, fm=FM_RUSNING, em=EM_RUSNING)
print(f'Trafikdygn: {datum} ({veckodag})  ·  {len(stat)} linjer')

### Översiktstabell över linjerna

In [ ]:
rader = []
for rid in gtfs['routes']['route_id']:
    s = stat[rid]
    fm = max((s['per_dir'][d]['fm_per_h'] for d in s['per_dir']), default=0)
    em = max((s['per_dir'][d]['em_per_h'] for d in s['per_dir']), default=0)
    rader.append({'route_id': rid, 'linje': s['kort'], 'namn': s['lang'],
                  'avg/dag': s['avg_dag'], 'riktning': 'båda' if s['bada_riktningar'] else 'en',
                  'FM/h (max)': fm, 'EM/h (max)': em})
linjetabell = pd.DataFrame(rader)
linjetabell

## Steg 3 – Välj linjer och rita kartan
Kryssa i linjerna du vill arbeta vidare med. Raden **"Valda linjer"** uppdateras direkt så du ser exakt vad som är
markerat (inget Ctrl-klick behövs). Klicka sedan **🗺️ Rita karta** – hela nätet visas, dina valda linjer framhävs,
och du kan klicka på en linje för statistik. Saknas `ipywidgets` kan du sätta `VALDA_LINJER` manuellt.

In [ ]:
VALDA_LINJER = []          # uppdateras av kryssrutorna nedan (eller sätt manuellt, t.ex. ["1","4"])
kryssrutor = {}

def rita_valda_karta(valda):
    if HAR_FOLIUM:
        return rita_natverk(gtfs, stat, valda=valda, filnamn=os.path.join(UT, 'gtfs_natverk.html'))
    rita_natverk_statisk(gtfs, stat, valda=valda, filnamn=os.path.join(UT, 'gtfs_natverk.png'))
    return None

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    kryssrutor = {rid: widgets.Checkbox(value=False, indent=False,
                     description=f"Linje {stat[rid]['kort']} – {stat[rid]['lang']}",
                     layout=widgets.Layout(width='auto'))
                  for rid in gtfs['routes']['route_id']}
    _status = widgets.HTML()
    _ut = widgets.Output()

    def _uppdatera(_=None):
        global VALDA_LINJER
        VALDA_LINJER = [rid for rid, cb in kryssrutor.items() if cb.value]
        namn = ', '.join('Linje ' + stat[r]['kort'] for r in VALDA_LINJER)
        _status.value = ("<b>Valda linjer:</b> " + (namn if VALDA_LINJER else '(inga – hela nätet visas)')
                         + f" &nbsp;<span style='color:#666'>({len(VALDA_LINJER)} st)</span>")
    for cb in kryssrutor.values():
        cb.observe(_uppdatera, names='value')
    _uppdatera()

    _btn = widgets.Button(description='🗺️ Rita karta', button_style='primary')
    def _klick(_):
        with _ut:
            clear_output(wait=True)
            print('Ritar karta – valda linjer:', VALDA_LINJER if VALDA_LINJER else '(hela nätet)')
            k = rita_valda_karta(VALDA_LINJER)
            if k is not None:
                display(k)
    _btn.on_click(_klick)

    display(widgets.HTML("<b>Kryssa i linjerna du vill arbeta vidare med:</b>"))
    display(widgets.VBox(list(kryssrutor.values())))
    display(_status)
    display(_btn)
    display(_ut)
except Exception as e:
    print('ipywidgets ej tillgängligt (', e, ') – sätt VALDA_LINJER manuellt (t.ex. ["1","4"]) och kör om cellen.')
    rita_valda_karta(VALDA_LINJER)
    print('Karta sparad i mappen', UT, '(gtfs_natverk.html/png).')

## Steg 4 – Spara urvalet
De valda linjerna (från kryssrutorna) sparas för det fortsatta arbetet.

In [ ]:
urval = VALDA_LINJER if VALDA_LINJER else list(gtfs['routes']['route_id'])
urval_df = pd.DataFrame({'route_id': urval,
                         'linje': [stat[r]['kort'] for r in urval],
                         'namn': [stat[r]['lang'] for r in urval]})
urval_df.to_csv(os.path.join(UT, 'valda_linjer.csv'), index=False)
print(f'Sparade {len(urval)} linjer till valda_linjer.csv'
      + ('' if VALDA_LINJER else ' (inga kryssade – sparade hela nätet)') + ':')
urval_df

# Del 2 – Optimering med genetisk algoritm (PAUSAD)

Den här delen är **pausad**. Den kör den genetiska algoritmen (bygger OSRM-restider, rekonstruerar dagens nät,
optimerar). Sätt `KOR_OPTIMERING = True` i cellen nedan **och** kör cellerna i Del 2 för att aktivera den.
Vid *Kör alla* stannar körningen här så att den tunga optimeringen inte startar av misstag.

In [ ]:
KOR_OPTIMERING = False   # <-- sätt True för att köra optimeringsdelen nedan
assert KOR_OPTIMERING, ('Del 2 (optimering) är pausad. Sätt KOR_OPTIMERING = True ovan och kör Del 2-cellerna '
                        'för att aktivera den. (Del 1 ovan påverkas inte.)')

## 1. Parametrar
Ändra dessa värden inför en körning. Sökvägarna utgår från att notebooken ligger i `-land/notebooks/`
och att indatafilerna ligger i `-land/data/`.

In [ ]:
from pathlib import Path

# --- Hitta repo-mappen automatiskt (fungerar oavsett var kärnan startas ifrån) ---
# Vi letar efter mappen som innehåller 'data/hallplatser.geojson'. Notebooken kan köras från notebooks/,
# från repo-roten (-land/), en nivå ovanför, ELLER från t.ex. Google Colab där repot klonats till en
# undermapp som /content/-land/ (då söker vi även i undermappar).
def hitta_basmapp():
    mal = Path('data') / 'hallplatser.geojson'
    def traff(bas):
        try:
            return (bas / mal).exists()
        except Exception:
            return False
    # 1) snabba kandidater: cwd, förälder, farförälder
    for bas in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if traff(bas):
            return bas.resolve()
    # 2) undermappar (djup 1–2) under cwd och dess förälder – fångar t.ex. /content/-land/
    for rot in [Path.cwd(), Path.cwd().parent]:
        try:
            for barn in rot.iterdir():
                if barn.is_dir():
                    if traff(barn):
                        return barn.resolve()
                    try:
                        for barnbarn in barn.iterdir():
                            if barnbarn.is_dir() and traff(barnbarn):
                                return barnbarn.resolve()
                    except Exception:
                        pass
        except Exception:
            pass
    # 3) gå uppåt i katalogträdet
    p = Path.cwd().resolve()
    for _ in range(8):
        if traff(p):
            return p
        if p.parent == p:
            break
        p = p.parent
    return Path.cwd().resolve()                 # sista utväg (ger tydligt fel + tips nedan)

# Sätt BAS_DIR manuellt här om din datamapp ligger på en annan plats. Exempel:
#   BAS_DIR = Path('/content/-land')          # Google Colab efter: !git clone .../-land
#   BAS_DIR = Path('/content/drive/MyDrive/-land')   # om repot ligger på Google Drive
BAS_DIR = hitta_basmapp()
if not (BAS_DIR / 'data' / 'hallplatser.geojson').exists():
    print('VARNING: hittade inte data/hallplatser.geojson (utgick från', str(BAS_DIR) + ').')
    print('  Sätt BAS_DIR manuellt ovan till mappen som innehåller data/ (repo-roten "-land").')
    print('  Tips: kör i en cell   !find / -name hallplatser.geojson 2>/dev/null   för att hitta sökvägen.')
else:
    print('Basmapp (repo):', BAS_DIR)

# --- Sökvägar ---
DATA_DIR       = str(BAS_DIR / 'data')
OUTPUT_DIR     = str(BAS_DIR / 'output')
GEOJSON_HPL    = os.path.join(DATA_DIR, 'hallplatser.geojson')
CSV_CENTROIDER = os.path.join(DATA_DIR, 'centroid_nyckel.csv')
XLSX_OD        = os.path.join(DATA_DIR, 'OD_matris.xlsx')
CSV_OD_CACHE   = os.path.join(DATA_DIR, 'od_long.csv')            # gles OD-cache (skapas)
CSV_RESTIDER   = os.path.join(OUTPUT_DIR, 'restider_hallplatser.csv')   # OSRM-cache (skapas)
CSV_RESULTAT   = os.path.join(OUTPUT_DIR, 'basta_linjenat.csv')   # resultat (skapas)
PNG_KARTA      = os.path.join(OUTPUT_DIR, 'linjenat_karta.png')

# --- Nätverksparametrar ---
ANTAL_LINJER        = 6            # antal busslinjer (din parameter; 6 = matchar dagens nät)
LINJE_LANGD_MIN_KM  = 6.0          # tillåtet längdspann per linje (km)
LINJE_LANGD_MAX_KM  = 60.0         # Åland är stort (Eckerö/Vårdö ligger långt bort)
CENTRAL_NAMN        = 'Bussplan'                 # central hållplats – matchas mot namn
CENTRAL_KOORD       = (19.9422816, 60.1022269)   # reserv om namnet inte hittas (Mariehamn, Bussplan)

# --- Restidsmodell ---
# Turtäthet (min). Skalär => samma för alla linjer; lista (längd ANTAL_LINJER) => per linje.
# Standard matchar dagens nät: linje 1–5 varje timme, linje 6 (skollinje) en gång på EM ~ lång headway.
HEADWAY_MIN         = [60, 60, 60, 60, 60, 120]
BYTESSTRAFF_MIN     = 5.0          # +5 min per byte
VANTETIDSVIKT       = 2.0          # varje väntad minut = 2 upplevda minuter
STRAFF_OBETJANAD    = 120.0        # straff (upplevda min) per enhet efterfrågan som saknar förbindelse
GANG_RADIE_M        = 600.0        # resenärer kan gå upp till så här långt till en trafikerad hållplats
GANG_HAST_KMH       = 4.5          # antagen gånghastighet

# --- OSRM ---
ANVAND_OSRM         = True
OSRM_BASE_URL       = 'https://router.project-osrm.org'   # egen server: t.ex. 'http://localhost:5000'
OSRM_MAX_TABLE      = 100          # max antal koordinater per /table-anrop (publik demo = 100)
FALLBACK_HASTIGHET  = 45.0         # km/h – används bara om OSRM inte kan nås

# --- Aggregering & GA ---
KLUSTER_M           = 40.0         # slå ihop hållplatser inom detta avstånd (m); 0 => ingen hopslagning
ANTAL_GRANNAR       = 10           # antal närmaste grannar som linjer kan gå vidare till
POP_STORLEK         = 40
GENERATIONER        = 90
MUTATIONSGRAD       = 0.35
SLUMPFRO            = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
rng = random.Random(SLUMPFRO)
np.random.seed(SLUMPFRO)
print('Parametrar laddade.')

## 2. Geografiska hjälpfunktioner

In [ ]:
JORDRADIE_M = 6371000.0  # jordens medelradie (m)

def haversine_m(lon1, lat1, lon2, lat2):
    la1, la2 = math.radians(lat1), math.radians(lat2)
    dla = math.radians(lat2 - lat1); dlo = math.radians(lon2 - lon1)
    h = math.sin(dla/2)**2 + math.cos(la1)*math.cos(la2)*math.sin(dlo/2)**2
    return 2*JORDRADIE_M*math.asin(math.sqrt(h))

def haversine_matris_m(lons, lats):
    """Vektoriserad NxN haversine (meter)."""
    lons = np.radians(np.asarray(lons, float)); lats = np.radians(np.asarray(lats, float))
    dlon = lons[None, :] - lons[:, None]
    dlat = lats[None, :] - lats[:, None]
    a = np.sin(dlat/2)**2 + np.cos(lats)[:, None]*np.cos(lats)[None, :]*np.sin(dlon/2)**2
    return 2*JORDRADIE_M*np.arcsin(np.sqrt(np.clip(a, 0, 1)))

## 3. Inläsning av indata
Hållplatser läses från GeoJSON, centroider från CSV och OD-matrisen från Excel. OD-matrisen är stor
(1477×1477) men gles, så den läses en gång till gles long-form `[origin_zon, dest_zon, efterfragan]` och cachas.

In [ ]:
def las_hallplatser(path):
    d = json.load(open(path, encoding='utf-8'))
    rader = []
    for f in d['features']:
        lon, lat = f['geometry']['coordinates']
        p = f['properties']
        namn = p.get('name') or p.get('name:sv') or p.get('description') or 'Namnlös'
        pid = p.get('@id') or f.get('id')
        rader.append({'platform_id': pid, 'namn': namn, 'lon': float(lon), 'lat': float(lat)})
    return pd.DataFrame(rader)

def las_centroider(path):
    df = pd.read_csv(path)
    df = df.rename(columns={'zon_id': 'zon_id', 'lon': 'lon', 'lat': 'lat'})
    return df

def las_od_matris(xlsx_path, cache_csv):
    """Läser OD-matrisen (1477x1477) till gles long-form och cachar."""
    if os.path.exists(cache_csv):
        return pd.read_csv(cache_csv)
    import openpyxl
    wb = openpyxl.load_workbook(xlsx_path, read_only=True, data_only=True)
    ws = wb['OD_matris']
    rader = ws.iter_rows(values_only=True)
    header = next(rader)
    dest_ids = list(header[1:])
    poster = []
    for row in rader:
        origin = row[0]
        if origin is None:
            continue
        for j, val in enumerate(row[1:]):
            if val:  # hoppa None och 0
                poster.append((origin, dest_ids[j], float(val)))
    df = pd.DataFrame(poster, columns=['origin_zon', 'dest_zon', 'efterfragan'])
    df.to_csv(cache_csv, index=False)
    return df

# ------------------------------------------------

In [ ]:
hallplatser = las_hallplatser(GEOJSON_HPL)
centroider  = las_centroider(CSV_CENTROIDER)
od_long     = las_od_matris(XLSX_OD, CSV_OD_CACHE)

print(f'Hållplatser (plattformar): {len(hallplatser)}')
print(f'Centroider (zoner):        {len(centroider)}')
print(f'OD-par (nollskilda):       {len(od_long)}   total efterfrågan: {od_long["efterfragan"].sum():.2f}')
print(f'Ursprungszoner med efterfrågan:   {od_long["origin_zon"].nunique()}')
print(f'Destinationszoner med efterfrågan:{od_long["dest_zon"].nunique()}')
hallplatser.head()

## 4. Hållplatsaggregering
Många hållplatser är *riktningspar* (samma läge, båda färdriktningarna). Vi slår ihop hållplatser inom
`KLUSTER_M` meter till en nod. Det minskar nätverket och ger tydligare linjer. Sätt `KLUSTER_M = 0`
för att behålla alla plattformar.

In [ ]:
def klustra_hallplatser(stops_df, cluster_m):
    """Slår ihop hållplatser inom cluster_m meter (riktningspar). union-find."""
    n = len(stops_df)
    lons = stops_df['lon'].to_numpy(); lats = stops_df['lat'].to_numpy()
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[ra] = rb
    # grov filtrering via grad-tröskel, exakt haversine per kandidatpar
    grader = cluster_m / 111320.0
    order = np.argsort(lats)
    for ii in range(n):
        i = order[ii]
        for jj in range(ii+1, n):
            j = order[jj]
            if lats[j] - lats[i] > grader:  # sorterad på lat -> kan bryta
                break
            if abs(lons[j]-lons[i]) > grader/max(math.cos(math.radians(lats[i])), 1e-6):
                continue
            if haversine_m(lons[i], lats[i], lons[j], lats[j]) <= cluster_m:
                union(i, j)
    grupper = {}
    for i in range(n):
        grupper.setdefault(find(i), []).append(i)
    kluster = []
    platform2kluster = {}
    for k, (root, idxs) in enumerate(sorted(grupper.items())):
        kid = f'H{k+1:04d}'
        sub = stops_df.iloc[idxs]
        # representativ koordinat = medel; namn = vanligast förekommande icke-namnlös
        namnrakning = sub['namn'][sub['namn'] != 'Namnlös']
        namn = namnrakning.mode().iloc[0] if len(namnrakning) else sub['namn'].iloc[0]
        kluster.append({'hpl_id': kid, 'namn': namn,
                        'lon': float(sub['lon'].mean()), 'lat': float(sub['lat'].mean()),
                        'antal_plattformar': len(idxs)})
        for pid in sub['platform_id']:
            platform2kluster[pid] = kid
    return pd.DataFrame(kluster), platform2kluster

# ------------------------------------------------

In [ ]:
if KLUSTER_M and KLUSTER_M > 0:
    hpl, platform2kluster = klustra_hallplatser(hallplatser, KLUSTER_M)
else:
    hpl = hallplatser.rename(columns={'platform_id': 'hpl_id'}).copy()
    hpl['antal_plattformar'] = 1
    platform2kluster = {p: p for p in hallplatser['platform_id']}

hpl = hpl.reset_index(drop=True)
N = len(hpl)
hpl_index = {h: i for i, h in enumerate(hpl['hpl_id'])}     # hpl_id -> radindex
print(f'Antal hållplatser efter aggregering: {N}')
hpl.head()

### Central hållplats
Den hållplats som **alla** linjer måste passera. Väljs i första hand på namn (`CENTRAL_NAMN`),
annars den hållplats som ligger närmast `CENTRAL_KOORD`.

In [ ]:
def hitta_central(hpl_df, namn, koord):
    traff = hpl_df[hpl_df['namn'].str.contains(namn, case=False, na=False)]
    if len(traff):
        # välj den kandidat som har flest plattformar (troligast huvudhållplatsen)
        return int(traff.sort_values('antal_plattformar', ascending=False).index[0])
    avst = haversine_vekt(koord[0], koord[1], hpl_df['lon'].to_numpy(), hpl_df['lat'].to_numpy())
    return int(np.argmin(avst))

central = hitta_central(hpl, CENTRAL_NAMN, CENTRAL_KOORD)
print(f'Central hållplats: idx={central}  id={hpl.loc[central, "hpl_id"]}  '
      f'namn="{hpl.loc[central, "namn"]}"  ({hpl.loc[central, "lon"]:.5f}, {hpl.loc[central, "lat"]:.5f})')

## 5. Restidsmatris via OSRM (med cache)
Åktiden mellan alla hållplatspar hämtas från OSRM (`/table`, bil-profil) och sparas i tydligt long-format
`from_id,to_id,duration_s,distance_m`. **Kontroll före körning:** finns en komplett cache för de aktuella
hållplatserna återanvänds den; saknas någon hållplats beräknas matrisen om automatiskt.

Om OSRM inte kan nås (t.ex. ingen internet/server) används en enkel **fallback** (fågelvägen × omvägsfaktor
delat på antagen hastighet) så att notebooken ändå kan köras – byt till en riktig OSRM-server för skarpa resultat.

In [ ]:
def bygg_restidsmatris_fallback(hpl_df, hastighet_kmh=40.0, omvag=1.3):
    lons = hpl_df['lon'].to_numpy(); lats = hpl_df['lat'].to_numpy()
    dist_km = haversine_matris_m(lons, lats)/1000.0*omvag
    dur_min = dist_km/hastighet_kmh*60.0
    np.fill_diagonal(dur_min, 0.0); np.fill_diagonal(dist_km, 0.0)
    return dur_min, dist_km

def _osrm_table_block(base, coords, src_idx, dst_idx, timeout=180):
    coord_str = ';'.join(f'{lo:.6f},{la:.6f}' for lo, la in coords)
    params = {'annotations': 'duration,distance',
              'sources': ';'.join(map(str, src_idx)),
              'destinations': ';'.join(map(str, dst_idx))}
    r = requests.get(f'{base}/table/v1/driving/{coord_str}', params=params, timeout=timeout)
    r.raise_for_status(); j = r.json()
    if j.get('code') != 'Ok':
        raise RuntimeError(f"OSRM: {j.get('code')} {j.get('message')}")
    return np.array(j['durations'], float), np.array(j['distances'], float)

def bygg_restidsmatris_osrm(hpl_df, base, max_table=100, paus=1.0, retries=4):
    coords = list(zip(hpl_df['lon'], hpl_df['lat'])); n = len(coords)
    b = max(1, max_table//2)
    DUR = np.full((n, n), np.nan); DIS = np.full((n, n), np.nan)
    blocks = [list(range(i, min(i+b, n))) for i in range(0, n, b)]
    for Si in blocks:
        for Dj in blocks:
            cc = [coords[k] for k in Si] + [coords[k] for k in Dj]
            si = list(range(len(Si))); di = list(range(len(Si), len(Si)+len(Dj)))
            for attempt in range(retries):
                try:
                    dur, dis = _osrm_table_block(base, cc, si, di); break
                except Exception:
                    if attempt == retries-1: raise
                    time.sleep(paus*(2**attempt))
            DUR[np.ix_(Si, Dj)] = dur; DIS[np.ix_(Si, Dj)] = dis
            time.sleep(paus)
    return DUR/60.0, DIS/1000.0

def spara_restider_csv(path, hpl_ids, dur_min, dist_km):
    n = len(hpl_ids); ids = np.array(hpl_ids)
    ii, jj = np.meshgrid(range(n), range(n), indexing='ij')
    pd.DataFrame({'from_id': ids[ii.ravel()], 'to_id': ids[jj.ravel()],
                  'duration_s': (dur_min*60).ravel(), 'distance_m': (dist_km*1000).ravel()}
                 ).to_csv(path, index=False)

def ladda_restider_csv(path, hpl_ids):
    """Returnerar (dur_min, dist_km) om cachen täcker ALLA hpl_ids, annars None."""
    if not os.path.exists(path): return None
    df = pd.read_csv(path)
    nuvarande = set(hpl_ids)
    if not nuvarande.issubset(set(df['from_id']) | set(df['to_id'])):
        return None
    idx = {h: i for i, h in enumerate(hpl_ids)}; n = len(hpl_ids)
    dur = np.full((n, n), np.nan); dis = np.full((n, n), np.nan)
    for f, t, ds, dm in df[['from_id', 'to_id', 'duration_s', 'distance_m']].itertuples(index=False):
        if f in idx and t in idx:
            dur[idx[f], idx[t]] = ds/60.0; dis[idx[f], idx[t]] = dm/1000.0
    if np.isnan(dur).any(): return None
    return dur, dis

def hamta_restider(hpl_df, csv_path, anvand_osrm=True, base=None, max_table=100):
    """Kontroll-först: använd cache om komplett, annars beräkna om automatiskt."""
    hpl_ids = hpl_df['hpl_id'].tolist()
    laddat = ladda_restider_csv(csv_path, hpl_ids)
    if laddat is not None:
        print(f'Restider: cache OK ({len(hpl_ids)} hållplatser).')
        return laddat
    print('Restider: cache saknas/ofullständig -> beräknar på nytt.')
    if anvand_osrm and base:
        try:
            dur, dis = bygg_restidsmatris_osrm(hpl_df, base, max_table=max_table)
            if np.isnan(dur).any():
                print('  Varning: OSRM gav luckor -> fyller med fallback.')
                fdur, fdis = bygg_restidsmatris_fallback(hpl_df)
                m = np.isnan(dur); dur[m] = fdur[m]; dis[m] = fdis[m]
        except Exception as e:
            print(f'  OSRM misslyckades ({e}) -> fallback (haversine).')
            dur, dis = bygg_restidsmatris_fallback(hpl_df)
    else:
        print('  Använder fallback-matris (haversine).')
        dur, dis = bygg_restidsmatris_fallback(hpl_df)
    spara_restider_csv(csv_path, hpl_ids, dur, dis)
    return dur, dis

# ------------------------------------------------

In [ ]:
dur_min, dist_km = hamta_restider(hpl, CSV_RESTIDER,
                                  anvand_osrm=ANVAND_OSRM, base=OSRM_BASE_URL, max_table=OSRM_MAX_TABLE)
print(f'Restidsmatris: {dur_min.shape}   '
      f'median åktid = {np.median(dur_min[dur_min>0]):.1f} min   max = {dur_min.max():.1f} min')

## 6. Koppla efterfrågan till hållplatser
Varje zon knyts till sin närmaste hållplats. Zon-OD:n aggregeras därmed till en **hållplats-OD-matris**
(gles), som GA:n väger restiderna mot.

In [ ]:
def koppla_zoner_till_hpl(centroider, hpl_df):
    hlon = hpl_df['lon'].to_numpy(); hlat = hpl_df['lat'].to_numpy()
    ids = hpl_df['hpl_id'].to_numpy()
    res = {}
    for _, z in centroider.iterrows():
        d = haversine_vekt(z['lon'], z['lat'], hlon, hlat)
        res[z['zon_id']] = ids[int(np.argmin(d))]
    return res

def haversine_vekt(lon, lat, lons, lats):
    la1 = math.radians(lat); la2 = np.radians(lats)
    dla = la2 - la1; dlo = np.radians(lons - lon)
    a = np.sin(dla/2)**2 + math.cos(la1)*np.cos(la2)*np.sin(dlo/2)**2
    return 2*JORDRADIE_M*np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def bygg_hpl_od(od_long, zon2hpl, hpl_index):
    """Aggregerar zon-OD till hållplats-OD (index-baserat)."""
    o = od_long['origin_zon'].map(zon2hpl).map(hpl_index)
    d = od_long['dest_zon'].map(zon2hpl).map(hpl_index)
    w = od_long['efterfragan'].to_numpy()
    mask = o.notna() & d.notna()
    o = o[mask].to_numpy().astype(int); d = d[mask].to_numpy().astype(int); w = w[mask.to_numpy()]
    df = pd.DataFrame({'o': o, 'd': d, 'w': w}).groupby(['o', 'd'], as_index=False)['w'].sum()
    return df

# ------------------------------------------------

In [ ]:
zon2hpl = koppla_zoner_till_hpl(centroider, hpl)
hpl_od  = bygg_hpl_od(od_long, zon2hpl, hpl_index)

print(f'Hållplats-OD-par: {len(hpl_od)}   total efterfrågan: {hpl_od["w"].sum():.2f}')
print(f'Unika ursprungshållplatser: {hpl_od["o"].nunique()}')

## 7. Genetisk algoritm

**Representation.** En individ = en lista av `ANTAL_LINJER` linjer. Varje linje är en sekvens av hållplatser
som **innehåller centralen** och vars längd ligger i `[LINJE_LANGD_MIN_KM, LINJE_LANGD_MAX_KM]`.

**Linjebygge.** Varje linje byggs som två armar ut från centralen mot var sitt *efterfrågeankare*
(en hållplats slumpad vägt efter `efterfrågan × avstånd från centralen`), genom att stega till närliggande
hållplatser i målets riktning. Viktningen får armarna att sträva utåt så att linjerna blir radiella och når
resenärer längre bort – likt ett verkligt Ålandsnät med nav i Mariehamn.

**Målfunktion (minimeras).** Upplevd restid för en resa =
`åktid (OSRM) + Σ byten·5 min + Σ påstigningar·(2 · headway/2)`.
Detta modelleras i en **linjegraf** där varje nod är `(hållplats, linje)`:
- åkkant inom en linje = OSRM-åktid,
- påstigningskant från resenärens start = upplevd väntetid för linjen,
- byteskant vid gemensam hållplats = `5 + upplevd väntetid för den nya linjen`.

Kortaste vägen beräknas med Dijkstra (scipy, multi-source: en sökning per ursprungshållplats). Objektivet är
efterfrågeviktad medelrestid plus ett straff för efterfrågan som saknar förbindelse.

In [ ]:
def bygg_grannar(dur_min, k):
    n = dur_min.shape[0]; g = []
    for i in range(n):
        d = dur_min[i].copy(); d[i] = np.inf
        g.append([int(x) for x in np.argsort(d)[:k]])
    return g

def linjelangd(line, dist_km):
    return sum(dist_km[line[i], line[i+1]] for i in range(len(line)-1))

def bygg_gang_grannar(hpl, radie_m=600.0, gang_hast_kmh=4.5):
    """För varje hållplats: närliggande hållplatser inom radie_m (index + gångtid i min).
    Modellerar att resenärer kan gå en kort bit till/från en hållplats som trafikeras av en linje."""
    lons = hpl['lon'].to_numpy(); lats = hpl['lat'].to_numpy()
    D = haversine_matris_m(lons, lats)
    gang_idx = []; gang_w = []
    for i in range(len(hpl)):
        j = np.where(D[i] <= radie_m)[0]            # D[i,i]=0 => sig själv ingår (gångtid 0)
        gang_idx.append(j.astype(int))
        gang_w.append((D[i, j] / 1000.0 / gang_hast_kmh * 60.0).astype(float))
    return gang_idx, gang_w

def demand_per_hpl(hpl_od, N):
    """Total efterfrågan (in + ut) per hållplats — används för att styra linjer mot resenärer."""
    w = np.zeros(N)
    np.add.at(w, hpl_od['o'].to_numpy(), hpl_od['w'].to_numpy())
    np.add.at(w, hpl_od['d'].to_numpy(), hpl_od['w'].to_numpy())
    return w

def valj_ankare(hpl_vikt, rng):
    """Slumpar en 'ankarhållplats' vägt efter efterfrågan (dit en linjearm ska sträva)."""
    tot = hpl_vikt.sum()
    if tot <= 0:
        return rng.randrange(len(hpl_vikt))
    r = rng.random()*tot; ack = 0.0
    for i, v in enumerate(hpl_vikt):
        ack += v
        if ack >= r:
            return i
    return len(hpl_vikt)-1

def _vandra_mot(start, target, dur_min, dist_km, grannar, visited, budget_km, rng):
    """Girig geografisk vandring från start mot target längs närliggande hållplatser."""
    arm = []; cur = start; length = 0.0
    while length < budget_km and cur != target:
        cand = [s for s in grannar[cur] if s not in visited and s != cur]
        if not cand:
            break
        cand.sort(key=lambda s: dur_min[s, target])  # välj granne närmast målet
        nxt = cand[0] if rng.random() < 0.75 else rng.choice(cand[:min(3, len(cand))])
        length += dist_km[cur, nxt]; arm.append(nxt); visited.add(nxt); cur = nxt
    return arm

def bygg_linje(central, lmin, lmax, dur_min, dist_km, grannar, hpl_vikt, rng):
    """Bygger en linje som två armar från centralen mot var sitt efterfrågeankare."""
    target_len = rng.uniform(lmin, lmax)
    visited = {central}
    a1 = valj_ankare(hpl_vikt, rng); a2 = valj_ankare(hpl_vikt, rng)
    arm1 = _vandra_mot(central, a1, dur_min, dist_km, grannar, visited, target_len/2, rng)
    langd1 = sum(dist_km[central if i == 0 else arm1[i-1], arm1[i]] for i in range(len(arm1)))
    arm2 = _vandra_mot(central, a2, dur_min, dist_km, grannar, visited, target_len-langd1, rng)
    line = list(reversed(arm1)) + [central] + arm2
    if len(line) < 2:  # nödfall: minst en granne
        cand = [s for s in grannar[central] if s != central]
        if cand:
            line = [central, cand[0]]
    return laga_langd(line, central, lmin, lmax, dist_km)

def laga_langd(line, central, lmin, lmax, dist_km):
    line = list(line)
    while linjelangd(line, dist_km) > lmax and len(line) > 2:
        if line[0] != central:
            line = line[1:]
        elif line[-1] != central:
            line = line[:-1]
        else:
            break
    return line

def bygg_individ(central, antal_linjer, lmin, lmax, dur_min, dist_km, grannar, hpl_vikt, rng):
    return [bygg_linje(central, lmin, lmax, dur_min, dist_km, grannar, hpl_vikt, rng)
            for _ in range(antal_linjer)]

def mutera_linje(line, central, lmin, lmax, dist_km, grannar, rng):
    line = list(line)
    action = rng.choice(['lagg_v', 'lagg_h', 'ta_v', 'ta_h'])
    if action == 'lagg_v':
        cand = [s for s in grannar[line[0]] if s not in line]
        if cand: line.insert(0, rng.choice(cand))
    elif action == 'lagg_h':
        cand = [s for s in grannar[line[-1]] if s not in line]
        if cand: line.append(rng.choice(cand))
    elif action == 'ta_v' and len(line) > 2 and line[0] != central:
        line = line[1:]
    elif action == 'ta_h' and len(line) > 2 and line[-1] != central:
        line = line[:-1]
    return laga_langd(line, central, lmin, lmax, dist_km)

In [ ]:
class Utvarderare:
    """Bygger linjegraf och beräknar upplevd restid via scipy-Dijkstra (multi-source)."""
    def __init__(self, N, dur_min, hpl_od, headways_min, transfer_penalty, wait_weight, penalty_unserved,
                 gang_idx=None, gang_w=None):
        self.N = N; self.dur = dur_min
        self.transfer_penalty = transfer_penalty; self.penalty_unserved = penalty_unserved
        self.wait_weight = wait_weight
        self.headways = headways_min  # lista per linje (min)
        self.o_arr = hpl_od['o'].to_numpy(); self.d_arr = hpl_od['d'].to_numpy()
        self.w_arr = hpl_od['w'].to_numpy(); self.W = self.w_arr.sum()
        self.origin_stops = sorted(set(self.o_arr.tolist()))
        origin_row = {s: i for i, s in enumerate(self.origin_stops)}
        self.o_rows = np.array([origin_row[o] for o in self.o_arr])
        self.same = (self.o_arr == self.d_arr)
        # Gångåtkomst: gang_idx[s]/gang_w[s] = närliggande hållplatser man kan gå till (index + gångtid min).
        # Utan gångdata kan man bara kliva på/av vid exakt sin egen närmaste hållplats.
        if gang_idx is None:
            gang_idx = [np.array([s]) for s in range(N)]
            gang_w = [np.array([0.0]) for _ in range(N)]
        self.gang_idx = gang_idx; self.gang_w = gang_w
        self.dest_unika = sorted(set(self.d_arr.tolist()))

    def perc_wait(self, l):
        return self.wait_weight * (self.headways[l]/2.0)

    def _bygg_graf(self, routes):
        node_id = {}; stop_nodes = defaultdict(list); stops_here = defaultdict(list)
        for l, line in enumerate(routes):
            for s in line:
                nid = len(node_id); node_id[(s, l)] = nid
                stop_nodes[s].append(nid); stops_here[s].append(l)
        rows = []; cols = []; wts = []
        for l, line in enumerate(routes):
            for i in range(len(line)-1):
                a = node_id[(line[i], l)]; b = node_id[(line[i+1], l)]
                t = self.dur[line[i], line[i+1]]
                rows += [a, b]; cols += [b, a]; wts += [t, t]
        for s, ls in stops_here.items():
            if len(ls) < 2: continue
            for li in ls:
                for lj in ls:
                    if li == lj: continue
                    rows.append(node_id[(s, li)]); cols.append(node_id[(s, lj)])
                    wts.append(self.transfer_penalty + self.perc_wait(lj))
        nbase = len(node_id)
        src_ids = []
        for k, s in enumerate(self.origin_stops):
            src = nbase + k; src_ids.append(src)
            # gå från sin närmaste hållplats till en närliggande betjänad hållplats och stig på där
            for s2, w in zip(self.gang_idx[s], self.gang_w[s]):
                s2 = int(s2)
                for l in stops_here.get(s2, []):
                    rows.append(src); cols.append(node_id[(s2, l)])
                    wts.append(float(w) + self.perc_wait(l))
        nnodes = nbase + len(self.origin_stops)
        csr = csr_matrix((wts, (rows, cols)), shape=(nnodes, nnodes))
        return csr, src_ids, stop_nodes, node_id

    def utvardera(self, routes, return_predecessors=False):
        csr, src_ids, stop_nodes, node_id = self._bygg_graf(routes)
        out = dijkstra(csr, directed=True, indices=src_ids,
                       return_predecessors=return_predecessors)
        dist = out[0] if return_predecessors else out
        stop_dist = np.full((len(self.origin_stops), self.N), np.inf)
        for s, nids in stop_nodes.items():
            stop_dist[:, s] = dist[:, nids].min(axis=1)
        # gångåtkomst i destinationsänden: kliv av vid närliggande betjänad hållplats och gå sista biten
        dest_dist = np.full_like(stop_dist, np.inf)
        for d in self.dest_unika:
            nb = self.gang_idx[d]; wv = self.gang_w[d]
            dest_dist[:, d] = (stop_dist[:, nb] + wv[None, :]).min(axis=1)
        tt = dest_dist[self.o_rows, self.d_arr]
        tt = np.where(self.same, 0.0, tt)
        served = np.isfinite(tt)
        obet_w = self.w_arr[~served].sum(); serv_w = self.w_arr[served].sum()
        serv_cost = (self.w_arr[served]*tt[served]).sum()
        obj = (serv_cost + self.penalty_unserved*obet_w)/self.W
        metrik = {'objektiv': obj, 'obetjanad_andel': obet_w/self.W,
                  'medel_restid_betjanad': serv_cost/max(serv_w, 1e-9)}
        if return_predecessors:
            return obj, metrik, dist, out[1], stop_dist, node_id
        return obj, metrik

In [ ]:
def genetisk_algoritm(central, antal_linjer, lmin, lmax, dur_min, dist_km, grannar, hpl_vikt,
                      utv, headways, pop_size, generationer, mutationsgrad, rng, verbose=True,
                      fro=None):
    def ny_ind():
        return bygg_individ(central, antal_linjer, lmin, lmax, dur_min, dist_km, grannar, hpl_vikt, rng)
    def ny_linje():
        return bygg_linje(central, lmin, lmax, dur_min, dist_km, grannar, hpl_vikt, rng)
    cache = {}
    def nyckel(ind):
        return tuple(tuple(r) for r in ind)
    def poang(ind):
        k = nyckel(ind)
        if k not in cache:
            cache[k] = utv.utvardera(ind)[0]
        return cache[k]
    # startpopulation: ev. frö-individer (t.ex. dagens linjenät) + slumpade
    pop = [[list(r) for r in ind] for ind in (fro or []) if len(ind) == antal_linjer]
    pop += [ny_ind() for _ in range(max(0, pop_size - len(pop)))]
    historik = []
    for g in range(generationer):
        pop.sort(key=poang)
        basta = poang(pop[0]); historik.append(basta)
        if verbose:
            print(f'  Gen {g+1:3d}/{generationer}: bästa objektiv = {basta:.3f}')
        elit_n = max(2, pop_size//2 - 2)
        elit = pop[:elit_n] + [ny_ind() for _ in range(2)]
        ny_pop = list(elit)
        while len(ny_pop) < pop_size:
            p1, p2 = rng.choice(elit), rng.choice(elit)
            barn = [list(p1[i]) if rng.random() < 0.5 else list(p2[i]) for i in range(antal_linjer)]
            if rng.random() < mutationsgrad:
                nya = []
                for r in barn:
                    if rng.random() < 0.15:            # strukturell mutation: bygg om hela linjen
                        nya.append(ny_linje())
                    else:                               # lokal mutation: justera ändhållplats
                        nya.append(mutera_linje(r, central, lmin, lmax, dist_km, grannar, rng))
                barn = nya
            ny_pop.append(barn)
        pop = ny_pop
    pop.sort(key=poang)
    return pop[0], historik

def _rakna_byten(pred_row, target, id_node):
    """Räknar antal byten (linjebyten) längs den återuppbyggda vägen till target."""
    kedja = []; cur = target; guard = 0
    while cur >= 0 and guard < 1000000:
        kedja.append(cur); nxt = pred_row[cur]
        if nxt == cur:
            break
        cur = nxt; guard += 1
    kedja = kedja[::-1]
    byten = 0
    for i in range(len(kedja)-1):
        a = id_node.get(kedja[i]); b = id_node.get(kedja[i+1])
        if a and b and a[0] == b[0] and a[1] != b[1]:
            byten += 1
    return byten

def analysera_basta(routes, utv):
    """Detaljerad analys av bästa lösningen inkl. bytesfördelning (via predecessors).
    Returnerar (obj, metrik, byten_hist) där byten_hist mappar antal_byten -> efterfrågan.
    Nyckeln -1 = obetjänad efterfrågan."""
    obj, metrik, dist, pred, stop_dist, node_id = utv.utvardera(routes, return_predecessors=True)
    id_node = {v: k for k, v in node_id.items()}  # nid -> (stop, line)
    origin_row = {s: i for i, s in enumerate(utv.origin_stops)}
    stop_nodes = defaultdict(list)
    for (s, l), nid in node_id.items():
        stop_nodes[s].append(nid)
    byten_hist = defaultdict(float)  # antal_byten -> efterfrågan (-1 = obetjänad)
    for o, d, w, same in zip(utv.o_arr, utv.d_arr, utv.w_arr, utv.same):
        if same:
            byten_hist[0] += w; continue
        r = origin_row[o]
        # inkludera gångåtkomst i deständen: bästa slutnod bland d:s närliggande betjänade hållplatser
        basta_kostnad = np.inf; target = None
        for s2, gw in zip(utv.gang_idx[d], utv.gang_w[d]):
            for n in stop_nodes.get(int(s2), []):
                kostnad = dist[r, n] + gw
                if kostnad < basta_kostnad:
                    basta_kostnad = kostnad; target = n
        if target is None or not np.isfinite(basta_kostnad):
            byten_hist[-1] += w; continue
        byten_hist[_rakna_byten(pred[r], target, id_node)] += w
    return obj, metrik, byten_hist

### Funktioner för dagens linjenät och interaktiva kartor
Nedan definieras hjälpfunktioner för (a) att rekonstruera dagens Ålandslinjer från ändpunkter/korridorer
och (b) att rita zoombara Folium-kartor.

In [ ]:
def narmaste_hpl(lon, lat, hpl_df):
    d = haversine_vekt(lon, lat, hpl_df['lon'].to_numpy(), hpl_df['lat'].to_numpy())
    return int(np.argmin(d))

def bygg_knn_graf(dur_min, k):
    """Symmetrisk k-närmaste-grannar-graf (restid) för att hitta rimliga vägar mellan hållplatser."""
    n = dur_min.shape[0]
    rows = []; cols = []; wt = []
    for i in range(n):
        d = dur_min[i].copy(); d[i] = np.inf
        for j in np.argsort(d)[:k]:
            j = int(j)
            rows.append(i); cols.append(j); wt.append(float(dur_min[i, j]))
    return csr_matrix((wt, (rows, cols)), shape=(n, n))

def vag_mellan(knn_graf, a, b):
    """Kortaste vägen (nodsekvens) mellan a och b på kNN-grafen, annars None."""
    dist, pred = dijkstra(knn_graf, directed=False, indices=[a], return_predecessors=True)
    if not np.isfinite(dist[0, b]):
        return None
    path = []; cur = b
    while cur != a and cur >= 0:
        path.append(cur); cur = int(pred[0, cur])
    if cur != a:
        return None
    path.append(a)
    return path[::-1]

def bygg_linje_fran_ankare(ankare, knn_graf):
    """Bygger en linje som kortaste vägar mellan på varandra följande ankarhållplatser."""
    linje = [ankare[0]]
    for i in range(len(ankare)-1):
        v = vag_mellan(knn_graf, ankare[i], ankare[i+1])
        if v is None:
            linje.append(ankare[i+1])            # nödfall: rak koppling
        else:
            linje += v[1:]
    ut = []                                       # ta bort direkta dubbletter
    for s in linje:
        if not ut or ut[-1] != s:
            ut.append(s)
    return ut

# Dagens linjenät (Ålandstrafiken, EM) beskrivet som ankarpunkter (namn, lon, lat) + headway (min).
# Centralen (Mariehamn, Bussplan) läggs till automatiskt som startpunkt.
DAGENS_LINJER_DEF = [
    ('Linje 1: Mariehamn–Hammarland–Eckerö',
     [('Näfsby (Hammarland)', 19.7814, 60.1942), ('Kattby', 19.7414, 60.2164),
      ('Storby (Eckerö)', 19.5586, 60.2233)], 60),
    ('Linje 2: Mariehamn–Godby–Geta',
     [('Godby centrum', 19.9892, 60.2306), ('Geta centrum', 19.8470, 60.3746)], 60),
    ('Linje 3: Mariehamn–Godby–Saltvik',
     [('Godby centrum', 19.9892, 60.2306), ('Kvarnbo (Saltvik)', 20.0623, 60.2761)], 60),
    ('Linje 4: Mariehamn–Godby–Sund–Vårdö',
     [('Godby centrum', 19.9892, 60.2306), ('Kastelholm (Sund)', 20.0919, 60.2306),
      ('Vårdö kyrka', 20.3717, 60.2425)], 60),
    ('Linje 5: Mariehamn–Lemland–Lumparland',
     [('Lemlands kyrka', 20.0841, 60.0717), ('Klemetsby (Lumparland)', 20.2585, 60.1166),
      ('Långnäs', 20.2964, 60.1176)], 60),
    ('Linje 6: Mariehamn–Godby–Emkarby–Gölby (skollinje)',
     [('Gölby', 19.9461, 60.1941), ('Emkarby', 19.8783, 60.2107),
      ('Godby centrum', 19.9892, 60.2306)], 120),
]

def bygg_dagens_linjenat(hpl, dur_min, central, knn_k=15):
    """Konstruerar dagens 6 linjer som stopp-sekvenser + namn + headways."""
    knn = bygg_knn_graf(dur_min, knn_k)
    linjer = []; namn = []; headways = []
    for lnamn, punkter, hw in DAGENS_LINJER_DEF:
        ankare = [central] + [narmaste_hpl(lo, la, hpl) for (_, lo, la) in punkter]
        # ta bort ev. upprepade ankare i rad
        ank2 = []
        for a in ankare:
            if not ank2 or ank2[-1] != a:
                ank2.append(a)
        linjer.append(bygg_linje_fran_ankare(ank2, knn))
        namn.append(lnamn); headways.append(float(hw))
    return linjer, namn, headways

def rita_interaktiv_karta(linjer, hpl, central, titel, filnamn,
                          linjenamn=None, hpl_vikt=None):
    """Bygger en interaktiv Folium-karta (zoombar) och sparar den som HTML. Returnerar kartobjektet."""
    import folium
    m = folium.Map(location=[60.18, 20.05], zoom_start=10, tiles='OpenStreetMap', control_scale=True)
    # Lager med alla hållplatser (avstängt som standard)
    fg_alla = folium.FeatureGroup(name='Alla hållplatser', show=False)
    for i, r in hpl.iterrows():
        rad = 2
        if hpl_vikt is not None and hpl_vikt.max() > 0:
            rad = 2 + 8*float(hpl_vikt[i])/float(hpl_vikt.max())
        folium.CircleMarker([r['lat'], r['lon']], radius=rad, color='#888', weight=0,
                            fill=True, fill_color='#888', fill_opacity=0.5,
                            popup=str(r['namn'])).add_to(fg_alla)
    fg_alla.add_to(m)
    farger = ['#e6194B', '#3cb44b', '#4363d8', '#911eb4', '#f58231', '#469990',
              '#800000', '#000075', '#9A6324', '#808000']
    for i, linje in enumerate(linjer):
        etikett = linjenamn[i] if linjenamn else f'Linje {i+1}'
        color = farger[i % len(farger)]
        fg = folium.FeatureGroup(name=etikett, show=True)
        coords = [[hpl['lat'].iloc[s], hpl['lon'].iloc[s]] for s in linje]
        folium.PolyLine(coords, color=color, weight=4, opacity=0.85, tooltip=etikett).add_to(fg)
        for s in linje:
            folium.CircleMarker([hpl['lat'].iloc[s], hpl['lon'].iloc[s]], radius=3.5,
                                color=color, weight=1, fill=True, fill_color=color, fill_opacity=0.9,
                                popup=f"{etikett}<br>{hpl['namn'].iloc[s]}").add_to(fg)
        fg.add_to(m)
    folium.Marker([hpl['lat'].iloc[central], hpl['lon'].iloc[central]],
                  tooltip='Central: ' + str(hpl['namn'].iloc[central]),
                  icon=folium.Icon(color='black', icon='star', prefix='fa')).add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    titel_html = (f'<div style="position:fixed;top:10px;left:60px;z-index:9999;background:white;'
                  f'padding:6px 10px;border-radius:6px;font-family:sans-serif;font-size:14px;'
                  f'box-shadow:0 1px 4px rgba(0,0,0,.3)"><b>{titel}</b></div>')
    m.get_root().html.add_child(folium.Element(titel_html))
    m.save(filnamn)
    return m

def rita_statisk_karta(linjer, hpl, central, dist_km, titel, filnamn, linjenamn=None, hpl_vikt=None):
    """Statisk matplotlib-karta (reserv om Folium saknas)."""
    import matplotlib.pyplot as plt
    N = len(hpl)
    fig, ax = plt.subplots(figsize=(11, 10))
    if hpl_vikt is not None and hpl_vikt.max() > 0:
        storlek = 6 + 350*(hpl_vikt/hpl_vikt.max())
    else:
        storlek = np.full(N, 10.0)
    ax.scatter(hpl['lon'], hpl['lat'], s=storlek, c='lightgray', edgecolors='none', zorder=1)
    farger = plt.cm.tab10(np.linspace(0, 1, max(len(linjer), 1)))
    for i, linje in enumerate(linjer):
        lons = hpl['lon'].iloc[linje].to_numpy(); lats = hpl['lat'].iloc[linje].to_numpy()
        etikett = linjenamn[i] if linjenamn else f'Linje {i+1}'
        ax.plot(lons, lats, '-', color=farger[i % 10], lw=2, alpha=0.85, label=etikett)
        ax.scatter(lons, lats, s=12, color=farger[i % 10], zorder=3)
    ax.scatter([hpl['lon'].iloc[central]], [hpl['lat'].iloc[central]],
               marker='*', s=500, color='black', zorder=5, label='Central hållplats')
    ax.set_aspect(1/np.cos(np.radians(float(hpl['lat'].mean()))))
    ax.set_title(titel); ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud')
    ax.legend(loc='upper right', fontsize=8, framealpha=0.9)
    plt.tight_layout(); plt.savefig(filnamn, dpi=130); plt.show()

## 8. Dagens linjenät (referens och startpunkt)

Vi utgår från dagens sex Ålandstrafik-linjer (linje 1–5 varje timme, linje 6 = skollinje). Linjerna
rekonstrueras här från sina ändpunkter/orter (Eckerö, Geta, Saltvik, Sund–Vårdö, Lumparland, Emkarby–Gölby)
och ritas på en interaktiv karta. De används dels som **referens**, dels som **frö** åt den genetiska
algoritmen (så att det optimerade nätet aldrig blir sämre än dagens).

Vi lägger också till **gångåtkomst**: en resenär kan gå en kort bit (upp till `GANG_RADIE_M`) till närmaste
trafikerade hållplats – annars skulle glesa nät se orimligt dåliga ut.

> **Obs:** Rekonstruktionen fångar rätt korridorer och ändpunkter men inte den exakta hållplatsföljden inne
> i Mariehamn. Dagens *modellerade* andel obetjänad är därför en pessimistisk överskattning. För en exakt
> nulägesjämförelse: importera Ålandstrafikens GTFS (exakt hållplatsföljd + tidtabell) – se README.

In [ ]:
# Gångåtkomst till/från närliggande hållplatser
gang_idx, gang_w = bygg_gang_grannar(hpl, radie_m=GANG_RADIE_M, gang_hast_kmh=GANG_HAST_KMH)

# Efterfrågan per hållplats (för kartstorlek + ankarviktning)
hpl_vikt = demand_per_hpl(hpl_od, N)

# Dagens 6 linjer + deras headways
dagens_linjer, dagens_namn, dagens_headways = bygg_dagens_linjenat(hpl, dur_min, central, knn_k=15)

utv_dagens = Utvarderare(N, dur_min, hpl_od, dagens_headways,
                         transfer_penalty=BYTESSTRAFF_MIN, wait_weight=VANTETIDSVIKT,
                         penalty_unserved=STRAFF_OBETJANAD, gang_idx=gang_idx, gang_w=gang_w)
obj_dagens, m_dagens = utv_dagens.utvardera(dagens_linjer)

print('=== Dagens linjenät (modellerat) ===')
for l, namn in zip(dagens_linjer, dagens_namn):
    print(f'  {namn}\n     {len(l)} hållplatser, {linjelangd(l, dist_km):.0f} km')
print(f'\nObjektiv (upplevd medelrestid + straff): {obj_dagens:.1f} min')
print(f'Efterfrågeviktad medelrestid (betjänad): {m_dagens["medel_restid_betjanad"]:.0f} min')
print(f'Andel obetjänad (modell, pessimistisk):  {m_dagens["obetjanad_andel"]*100:.0f} %')

In [ ]:
# Interaktiv karta över dagens nät (faller tillbaka på statisk karta om Folium saknas)
try:
    karta_dagens = rita_interaktiv_karta(
        dagens_linjer, hpl, central, 'Dagens linjenät (Åland, EM) – rekonstruerat',
        os.path.join(OUTPUT_DIR, 'dagens_linjenat.html'), dagens_namn, hpl_vikt)
    print('Interaktiv karta sparad: output/dagens_linjenat.html')
except Exception as e:
    print('Folium saknas/fel (', e, ') – ritar statisk karta. Installera med: pip install folium')
    rita_statisk_karta(dagens_linjer, hpl, central, dist_km, 'Dagens linjenät (rekonstruerat)',
                       os.path.join(OUTPUT_DIR, 'dagens_linjenat.png'), dagens_namn, hpl_vikt)
    karta_dagens = None
karta_dagens

## 9. Kör den genetiska algoritmen (startar från dagens nät)
GA:n startar med dagens linjenät som frö och söker ett bättre nät med samma antal linjer och samma
turtätheter. Ankarviktningen (`efterfrågan × avstånd från centralen`) gör att linjerna når ut till
resenärer längre bort.

In [ ]:
# Turtätheter: matcha dagens om antalet linjer stämmer, annars använd HEADWAY_MIN.
if ANTAL_LINJER == len(dagens_headways):
    headways = list(dagens_headways)
elif isinstance(HEADWAY_MIN, (list, tuple)):
    assert len(HEADWAY_MIN) == ANTAL_LINJER, 'HEADWAY_MIN-listan måste ha längd ANTAL_LINJER'
    headways = [float(h) for h in HEADWAY_MIN]
else:
    headways = [float(HEADWAY_MIN)] * ANTAL_LINJER

grannar = bygg_grannar(dur_min, ANTAL_GRANNAR)
ankar_vikt = hpl_vikt * dist_km[central].copy()

utv = Utvarderare(N, dur_min, hpl_od, headways,
                  transfer_penalty=BYTESSTRAFF_MIN, wait_weight=VANTETIDSVIKT,
                  penalty_unserved=STRAFF_OBETJANAD, gang_idx=gang_idx, gang_w=gang_w)

fro = [dagens_linjer] if ANTAL_LINJER == len(dagens_linjer) else None   # starta från dagens nät
t0 = time.time()
basta, historik = genetisk_algoritm(
    central, ANTAL_LINJER, LINJE_LANGD_MIN_KM, LINJE_LANGD_MAX_KM,
    dur_min, dist_km, grannar, ankar_vikt, utv, headways,
    POP_STORLEK, GENERATIONER, MUTATIONSGRAD, rng, verbose=False, fro=fro)
print(f'Klart på {time.time()-t0:.1f} s. Bästa objektiv: {historik[0]:.2f} -> {historik[-1]:.2f}')

## 10. Resultat och jämförelse med dagens nät

In [ ]:
obj, metrik, byten_hist = analysera_basta(basta, utv)
tot_w = sum(byten_hist.values())

# Jämförelse dagens vs optimerat (samma turtätheter och modell)
jmf = pd.DataFrame({
    'Nyckeltal': ['Objektiv (min)', 'Medelrestid betjänad (min)', 'Obetjänad (%)'],
    'Dagens (modell)': [round(obj_dagens, 1), round(m_dagens['medel_restid_betjanad'], 1),
                        round(m_dagens['obetjanad_andel']*100, 1)],
    'Optimerat': [round(obj, 1), round(metrik['medel_restid_betjanad'], 1),
                  round(metrik['obetjanad_andel']*100, 1)],
})
print('=== Jämförelse (lägre är bättre) ===')
print(jmf.to_string(index=False))
print(f'\nFörbättring av objektiv jämfört med dagens: {(1-obj/obj_dagens)*100:.1f} %')

print('\nBytesfördelning i optimerat nät (andel av efterfrågan):')
for k in sorted(byten_hist):
    etikett = 'obetjänad' if k == -1 else f'{k} byten'
    print(f'  {etikett:>12}: {byten_hist[k]/tot_w*100:5.1f} %')

print('\n=== Optimerade linjer ===')
rader = []
for i, linje in enumerate(basta):
    rader.append({'linje': i+1, 'antal_hpl': len(linje), 'langd_km': round(linjelangd(linje, dist_km), 1),
                  'headway_min': headways[i],
                  'fran': hpl.loc[linje[0], 'namn'], 'till': hpl.loc[linje[-1], 'namn']})
linjer_df = pd.DataFrame(rader)
print(f'Total linjelängd: {linjer_df["langd_km"].sum():.1f} km')
linjer_df

## 11. Visualisering

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(historik)+1), historik, marker='o', ms=3)
plt.xlabel('Generation'); plt.ylabel('Bästa objektiv (upplevd restid, min)')
plt.title('GA-konvergens'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Interaktiv karta över det optimerade nätet (zooma/panorera; klicka på hållplatser för namn)
linjenamn_opt = [f'Linje {i+1} ({linjelangd(l, dist_km):.0f} km)' for i, l in enumerate(basta)]
try:
    karta_opt = rita_interaktiv_karta(
        basta, hpl, central, 'Optimerat busslinjenät – Åland (EM)',
        os.path.join(OUTPUT_DIR, 'optimerat_linjenat.html'), linjenamn_opt, hpl_vikt)
    print('Interaktiv karta sparad: output/optimerat_linjenat.html')
except Exception as e:
    print('Folium saknas/fel (', e, ') – ritar statisk karta. Installera med: pip install folium')
    rita_statisk_karta(basta, hpl, central, dist_km, 'Optimerat busslinjenät – Åland',
                       PNG_KARTA, linjenamn_opt, hpl_vikt)
    karta_opt = None
karta_opt

## 12. Export av bästa lösningen

In [ ]:
rader = []
for i, linje in enumerate(basta):
    for ordn, s in enumerate(linje):
        rader.append({'linje': i+1, 'ordning': ordn, 'hpl_id': hpl.loc[s, 'hpl_id'],
                      'namn': hpl.loc[s, 'namn'], 'lon': hpl.loc[s, 'lon'], 'lat': hpl.loc[s, 'lat']})
resultat_df = pd.DataFrame(rader)
resultat_df.to_csv(CSV_RESULTAT, index=False)
print(f'Bästa linjenät sparat: {CSV_RESULTAT}  ({len(resultat_df)} rader)')
resultat_df.head(12)